# Task 1: Multi-Agent System using LangGraph

This notebook demonstrates a reproducible implementation of a
multi-agent AI system using LangGraph.

Agents:
- Data Collector Agent (API-based data fetching)
- Analyst Agent (Gemini-based reasoning)

The system generates a company intelligence report.

### Importing required libraries

In [2]:
from typing import TypedDict, Any, Dict
import os
import requests

import google.generativeai as genai

from langgraph.graph import StateGraph, END
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

import warnings
warnings.filterwarnings('ignore')

C:\Users\Dell\AppData\Local\Temp\ipykernel_22812\3415399177.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


### Getting API KEYS

In [3]:
NEWS_API_KEY = os.getenv('NEWS_API')
APLHA_API_KEY = os.getenv('ALPHA_VANTAGE')
GEMINI_API_KEY = os.getenv('GOOGLE_API_KEY3')

### API Tools

In [4]:
def fetch_company_news(company:str):
    url = "https://newsapi.org/v2/everything"
    params = {
        'q': company,
        'language': 'en',
        'sortBy': 'publishedAt',
        'pageSize': 3,
        'apiKey': NEWS_API_KEY
    }
    
    response = requests.get(url, params=params)
    articles = response.json().get('articles', [])
    return [a['title'] for a in articles]


def fetch_stock_data(symbol:str):
    url = 'https://www.alphavantage.co/query'
    params = {
        'function': 'GLOBAL_QUOTE',
        'symbol': symbol,
        'apikey': APLHA_API_KEY
    }
    
    response = requests.get(url, params=params)
    return response.json().get('Global Quote', {})


def get_company_data(company:str):
    return {
        'company': company,
        'recent_news': fetch_company_news(company),
        'stock_data': fetch_stock_data(company)
    }

### Configuire LLM model

In [5]:
genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel('gemini-2.5-flash-lite',
                              generation_config={'temperature':0.2})

### Agent Definitions

In [6]:
def data_collector_agent(state: Dict):
    data = state['tool'](state['company'])
    return {'company_data': data}


def analyst_agent(state: Dict):
    prompt = ChatPromptTemplate.from_template('''
    You are a financial analyst.
    
    Company Data: {company_data}
    
    Provide:
    - Market Summary
    - Key insights
    - Potential risks
    ''')
    
    response = model.invoke(
        prompt.format_messages(company_data = state['company_data'])
    )
    
    return {'final_report': response.content}

## LangGraph Orchestrator

In [7]:
class AgentState(TypedDict):
    company: str
    tool: Any
    company_data: dict
    final_report: str


graph = StateGraph(AgentState)

graph.add_node("collector", data_collector_agent)
graph.add_node("analyst", analyst_agent)

graph.set_entry_point("collector")
graph.add_edge("collector", "analyst")
graph.add_edge("analyst", END)

app = graph.compile()

### Execute the Workflow

In [8]:
result = app.invoke({
    "company": "Tata Consultancy Services",
    "tool": get_company_data
})

print("===== FINAL COMPANY INTELLIGENCE REPORT =====\n")
print(result["final_report"])

AttributeError: 'GenerativeModel' object has no attribute 'invoke'